# Reference · Day 3 studio — make the winner change

**Not a marking key.** You picked your own three contenders, so your numbers are not
meant to match these. What is worth comparing is the *shape* of what you found.

Two things happened in the room. Some teams flipped their worst model into first place
in about ten seconds. Others searched three hundred seeds and could not do it. Both are
correct results, and they mean opposite things. This notebook runs one of each so you
can see which one you were holding.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from stat764 import load

ames = load("ames.csv")

NUMERIC = ["Gr_Liv_Area", "Lot_Area", "Year_Built", "Overall_Qual",
           "Total_Bsmt_SF", "Garage_Cars"]
CATEGORICAL = ["Neighborhood", "Central_Air"]

X = ames[NUMERIC + CATEGORICAL]
y = ames["SalePrice"]


def make(model):
    """The same pipeline the meeting used, with the estimator swapped in."""
    prep = ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("prep", prep), ("model", model)])


cv = KFold(n_splits=10, shuffle=True, random_state=764)
print(f"{len(X)} houses, {X.shape[1]} predictors")

## A helper, so the two cases are measured identically

Everything below runs through this. Writing the measurement **once** and swapping the
models is itself the habit: if the two cases were measured by two blocks of copied
code, any difference between them could be a difference in the code.

In [ ]:
def win_counts(models, n_splits=50):
    """Score every model on the same `n_splits` random splits. Return (scores, winners)."""
    scores = {name: [] for name in models}
    winners = []
    for seed in range(n_splits):
        A, B, a, b = train_test_split(X, y, test_size=0.25, random_state=seed)
        s = {name: r2_score(b, make(m).fit(A, a).predict(B)) for name, m in models.items()}
        for name, v in s.items():
            scores[name].append(v)
        winners.append(max(s, key=s.get))
    return scores, winners


def report(label, models):
    scores, winners = win_counts(models)
    table = pd.DataFrame({
        "wins (of 50)": pd.Series(winners).value_counts().reindex(models, fill_value=0),
        "mean R2": {k: np.mean(v) for k, v in scores.items()},
        "spread": {k: np.max(v) - np.min(v) for k, v in scores.items()},
    })
    print(f"\n{label}")
    print(table.round(3).to_string())
    between = table["mean R2"].max() - table["mean R2"].min()
    within = table["spread"].mean()
    print(f"  between the models: {between:.3f}   from the split alone: {within:.3f}"
          f"   ({within / between:.0f}x)")
    return scores


def hunt_for_a_seed(models, scores, limit=300):
    """Try to make the worst-on-average model come first on some single split."""
    worst = min(models, key=lambda k: np.mean(scores[k]))
    for seed in range(limit):
        A, B, a, b = train_test_split(X, y, test_size=0.25, random_state=seed)
        s = {name: r2_score(b, make(m).fit(A, a).predict(B)) for name, m in models.items()}
        if max(s, key=s.get) == worst:
            print(f"  seed {seed} puts {worst} first: "
                  + ", ".join(f"{k} {v:.3f}" for k, v in s.items()))
            return seed
    print(f"  {worst} never wins in {limit} seeds — it is genuinely behind")
    return None

## Case A — three models that are actually within noise of each other

In [ ]:
case_a = {
    "OLS": LinearRegression(),
    "Ridge(10)": Ridge(alpha=10),
    "Tree(d=6)": DecisionTreeRegressor(max_depth=6, random_state=0),
}
scores_a = report("Case A — OLS, Ridge(10), Tree(d=6)", case_a)

print("\ncan the worst one be made to look best?")
seed_a = hunt_for_a_seed(case_a, scores_a)

All three win a decent share of the fifty splits, and a flattering seed turns up almost
immediately. That is not a flaw in the search — it is the honest summary: **on this
evidence these three models are not distinguishable.** A single split cannot tell them
apart, so any single split that appears to is reporting the split.

## Case B — the same procedure, models that really do differ

In [ ]:
case_b = {
    "OLS": LinearRegression(),
    "Ridge(100)": Ridge(alpha=100),
    "kNN(k=10)": KNeighborsRegressor(n_neighbors=10),
}
scores_b = report("Case B — OLS, Ridge(100), kNN(k=10)", case_b)

print("\ncan the worst one be made to look best?")
seed_b = hunt_for_a_seed(case_b, scores_b)

Same code, opposite answer. One model wins almost every split and the search fails.

**That failure is a measurement, not a dead end.** You have just established that the
gap survives every partition you threw at it — which is a far stronger statement than
"my model had the best R-squared."

## The question that settles it: gap against fold-to-fold SD

In [ ]:
cv_means = {}
for label, models in [("A", case_a), ("B", case_b)]:
    print(f"\ncase {label}    {'10-fold CV':>22}{'fold SD':>9}")
    for name, m in models.items():
        folds = cross_val_score(make(m), X, y, cv=cv, scoring="r2")
        cv_means[(label, name)] = (folds.mean(), folds.std())
        print(f"  {name:<14}{folds.mean():>12.3f}{folds.std():>8.3f}")

for label in ("A", "B"):
    vals = {k[1]: v for k, v in cv_means.items() if k[0] == label}
    gap = max(v[0] for v in vals.values()) - min(v[0] for v in vals.values())
    sd = np.mean([v[1] for v in vals.values()])
    se = sd / np.sqrt(cv.get_n_splits())      # spread OF THE MEAN, not of one fold
    print(f"\ncase {label}: gap {gap:.3f}   fold SD {sd:.3f}   SE of the mean {se:.3f}")
    if gap < se:
        print("   the gap is inside the uncertainty of the estimate itself — no winner")
    elif gap < sd:
        print("   bigger than the SE but smaller than a single fold's spread — suggestive,")
        print("   not settled. Repeat the CV before you say anything stronger.")
    else:
        print("   clears both bars — the difference survives every partition tried")

That comparison — **gap against spread** — is the whole of today. It is the question to
ask of any reported difference between two models, including your own, for the rest of
the course.

⚠ There are two spreads and they answer different questions. The **fold SD** is how much
one fold's estimate bounces around; the **SE of the mean** is `SD / sqrt(k)`, how much the
averaged estimate would move if you re-ran the whole thing. The gap you care about is a
gap between *means*, so the SE is the honest comparison and the fold SD is the
deliberately conservative one. Case B clears the SE comfortably and only just clears the
fold SD — which is why "the difference is real" is a fair claim there and "it is large"
is not.

## What you would say, and what you would refuse to say

For **case A**, the defensible recommendation is: *any of these three; pick on cost,
interpretability or maintenance, because the data cannot separate them.* The thing to
refuse is naming a winner.

For **case B**: *kNN, and the margin is larger than the fold-to-fold variation.* The
thing to refuse is still a bare point estimate — report it with its spread.

⚠ Note what neither case licenses: **why** a model wins. Nothing here says anything
about which predictors matter or what causes a price. That is Week 10, and it is a
harder question than this one.

## Check your own notebook against this

Not "did I get these numbers". These:

| | |
|---|---|
| **1** | Did you score every model on the **same** splits, rather than re-splitting per model? If each model got its own partition, you measured partitions. |
| **2** | Did you report **how often** each model won, rather than only its mean? A mean hides a three-way tie. |
| **3** | When you hunted for a flattering seed, did you write down the outcome **either way**? "I searched and failed" is the result that earns you the right to claim a real difference. |
| **4** | Did you compare your gap to the **fold-to-fold SD** before calling anything a winner? |
| **5** | Does the notebook survive **Restart & Run All**? |

If 1 or 4 is a no, that is the one to fix — those two are what Lab 2 is assessing.
If 3 is a no, it costs you nothing to go back and record it, and it is the difference
between an anecdote and a measurement.

⚠ **The fastest way to be wrong this semester is to report one number from one split.**
You now have two separate ways to catch yourself doing it.